In [1]:
import duckdb
import polars as pl
import gc
import os
from pathlib import Path

INPUT_RAW = Path("/kaggle/input/notebooks/fedorfesenko/ranking-create-raw-feats")
INPUT_INTERACTIONS = Path("/kaggle/input/datasets/fedorfesenko/train-interactions")
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

print(f"RAW: {INPUT_RAW}")
print(f"INT: {INPUT_INTERACTIONS}")
print(f"WORK: {WORK}")

RAW: /kaggle/input/notebooks/fedorfesenko/ranking-create-raw-feats
INT: /kaggle/input/datasets/fedorfesenko/train-interactions
WORK: /kaggle/working


In [2]:
df_ranking_train = pl.read_parquet(INPUT_INTERACTIONS / "df_ranking_train.parquet")
df_ranking_val   = pl.read_parquet(INPUT_INTERACTIONS / "df_ranking_val.parquet")
df_test          = pl.read_parquet(INPUT_INTERACTIONS / "df_test.parquet")

print(f"df_ranking_train: {df_ranking_train.shape}")
print(f"df_ranking_val:   {df_ranking_val.shape}")
print(f"df_test:          {df_test.shape}")

df_ranking_train: (15769472, 14)
df_ranking_val:   (323736, 14)
df_test:          (292826, 11)


In [3]:
# 1. Выдачи с CartUpdate
requests_with_cart = (
    df_ranking_train
    .filter(pl.col("action_type") == "AT_CartUpdate")
    ["request_id"]
    .unique()
)
print(f"Requests with CartUpdate: {len(requests_with_cart)}")

df_train_filtered = df_ranking_train.filter(
    pl.col("request_id").is_in(requests_with_cart)
)
print(f"df_train_filtered: {df_train_filtered.shape}")

# 2. Топ-10 View по position
view_top10 = (
    df_train_filtered
    .filter(pl.col("action_type") == "AT_View")
    .with_columns(
        pl.col("position_in_request").rank("ordinal").over("request_id").alias("pos_rank")
    )
    .filter(pl.col("pos_rank") <= 10)
    .drop("pos_rank")
)
print(f"view_top10: {view_top10.shape}")

# 3. Топ-3 Click по position
click_top3 = (
    df_train_filtered
    .filter(pl.col("action_type") == "AT_Click")
    .with_columns(
        pl.col("position_in_request").rank("ordinal").over("request_id").alias("pos_rank")
    )
    .filter(pl.col("pos_rank") <= 3)
    .drop("pos_rank")
)
print(f"click_top3: {click_top3.shape}")

# 4. Все CartUpdate
cart_all = df_train_filtered.filter(pl.col("action_type") == "AT_CartUpdate")
print(f"cart_all: {cart_all.shape}")

# 5. Собираем
df_ranking_train_us = pl.concat([cart_all, view_top10, click_top3]).unique()
print(f"df_ranking_train_us: {df_ranking_train_us.shape}")
print(f"target mean: {df_ranking_train_us['target'].mean():.4f}")

df_ranking_train_us.write_parquet(WORK / "df_ranking_train_us.parquet")

del requests_with_cart, df_train_filtered, view_top10, click_top3, cart_all
gc.collect()
print("Saved df_ranking_train_us.parquet")

Requests with CartUpdate: 170829


/tmp/ipykernel_16/1987127263.py:10: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df_train_filtered = df_ranking_train.filter(


df_train_filtered: (6802675, 14)
view_top10: (1473690, 14)
click_top3: (35750, 14)
cart_all: (301129, 14)
df_ranking_train_us: (1810569, 14)
target mean: 0.1663
Saved df_ranking_train_us.parquet


In [4]:
# T_train_end — из train events
df_interactions_train = pl.read_parquet(INPUT_INTERACTIONS / "df_interactions_train.parquet")
df_interactions_val   = pl.read_parquet(INPUT_INTERACTIONS / "df_interactions_val.parquet")

T_TRAIN_END = df_interactions_train["timestamp"].max()
T_VAL_END   = df_interactions_val["timestamp"].max()

print(f"T_TRAIN_END = {T_TRAIN_END}")
print(f"T_VAL_END   = {T_VAL_END}")

# Train (undersampled) — свой timestamp
df_train_ft = df_ranking_train_us.with_columns(
    pl.lit("train").alias("split"),
    pl.col("timestamp").alias("feature_timestamp"),
)

# Val — timestamp на T_TRAIN_END
df_val_ft = df_ranking_val.with_columns(
    pl.lit("val").alias("split"),
    pl.lit(T_TRAIN_END, dtype=pl.Int64).alias("feature_timestamp"),
)

# Test — timestamp на T_VAL_END
df_test_ft = df_test.with_columns(
    pl.lit("test").alias("split"),
    pl.lit(T_VAL_END, dtype=pl.Int64).alias("feature_timestamp"),
)

print(f"df_train_ft: {df_train_ft.shape}")
print(f"df_val_ft:   {df_val_ft.shape}")
print(f"df_test_ft:  {df_test_ft.shape}")

del df_interactions_train, df_interactions_val
gc.collect()

T_TRAIN_END = 1706290114
T_VAL_END   = 1706895291
df_train_ft: (1810569, 16)
df_val_ft:   (323736, 16)
df_test_ft:  (292826, 13)


0

In [5]:
item_raw = pl.read_parquet(INPUT_RAW / "item_counters_raw.parquet")
item_unique = pl.read_parquet(INPUT_RAW / "item_unique_counters_raw.parquet")

print(f"item_raw:    {item_raw.shape}")
print(f"item_unique: {item_unique.shape}")

item_raw = item_raw.with_columns(
    pl.from_epoch(pl.col("ts"), time_unit="s").dt.date().alias("day")
)
item_unique = item_unique.with_columns(
    pl.col("day").cast(pl.Date)
)

item_counters_raw_v2 = item_raw.join(
    item_unique,
    on=["product_id", "day"],
    how="left",
)
print(f"item_counters_raw_v2: {item_counters_raw_v2.shape}")
item_counters_raw_v2.write_parquet(WORK / "item_counters_raw_v2.parquet")

del item_raw, item_unique, item_counters_raw_v2
gc.collect()
print("Saved item_counters_raw_v2.parquet")

item_raw:    (16508614, 20)
item_unique: (2874564, 6)
item_counters_raw_v2: (16508614, 25)
Saved item_counters_raw_v2.parquet


In [6]:
# User counters
df_train_ft.write_parquet(WORK / "_train_ft.parquet")
duckdb.sql(f"""
    COPY (
        SELECT
            r.user_id, r.product_id, r.request_id, r.split, r.feature_timestamp,
            u.n_views_all, u.n_clicks_all, u.n_cart_updates_all, u.n_purchases_all,
            u.n_views_30d, u.n_clicks_30d, u.n_cart_updates_30d, u.n_purchases_30d,
            u.n_views_7d,  u.n_clicks_7d,  u.n_cart_updates_7d,  u.n_purchases_7d,
            u.n_views_1d,  u.n_clicks_1d,  u.n_cart_updates_1d,  u.n_purchases_1d,
            u.days_since_first_action, u.days_since_last_action
        FROM '{WORK}/_train_ft.parquet' r
        ASOF LEFT JOIN '{INPUT_RAW}/user_counters_raw.parquet' u
        ON r.user_id = u.user_id
        AND r.feature_timestamp > u.ts
    ) TO '{WORK}/_train_user.parquet' (FORMAT PARQUET)
""")
print("user done")

duckdb.sql(f"""
    COPY (
        SELECT
            b.*,
            u.item_n_views_all, u.item_n_clicks_all, u.item_n_cart_updates_all, u.item_n_purchases_all,
            u.item_n_views_30d, u.item_n_clicks_30d, u.item_n_cart_updates_30d, u.item_n_purchases_30d,
            u.item_n_views_7d,  u.item_n_clicks_7d,  u.item_n_cart_updates_7d,  u.item_n_purchases_7d,
            u.item_n_views_1d,  u.item_n_clicks_1d,  u.item_n_cart_updates_1d,  u.item_n_purchases_1d,
            u.item_n_unique_users_all, u.item_n_unique_users_30d, u.item_n_unique_users_7d, u.item_n_unique_users_1d,
            u.item_days_since_first_view, u.item_days_since_last_view
        FROM '{WORK}/_train_user.parquet' b
        ASOF LEFT JOIN '{WORK}/item_counters_raw_v2.parquet' u
        ON b.product_id = u.product_id
        AND b.feature_timestamp > u.ts
    ) TO '{WORK}/_train_user_item.parquet' (FORMAT PARQUET)
""")
print("item done")

duckdb.sql(f"""
    COPY (
        SELECT
            b.*,
            u.ui_n_views_all, u.ui_n_clicks_all, u.ui_n_cart_updates_all, u.ui_n_purchases_all,
            u.ui_days_since_last_action, u.ui_days_since_last_cart_update, u.ui_days_since_last_purchase
        FROM '{WORK}/_train_user_item.parquet' b
        ASOF LEFT JOIN '{INPUT_RAW}/ui_counters_raw.parquet' u
        ON b.user_id = u.user_id AND b.product_id = u.product_id
        AND b.feature_timestamp > u.ts
    ) TO '{WORK}/_train_ui.parquet' (FORMAT PARQUET)
""")
print("ui done")

# UC через GROUP BY (быстрая дедупликация)
duckdb.sql(f"""
    COPY (
        WITH uc_joined AS (
            SELECT
                r.user_id, r.product_id, r.request_id,
                u.uc_n_views_all, u.uc_n_clicks_all, u.uc_n_cart_updates_all, u.uc_n_purchases_all,
                u.uc_n_views_30d, u.uc_n_clicks_30d, u.uc_n_cart_updates_30d, u.uc_n_purchases_30d,
                u.uc_n_views_7d,  u.uc_n_clicks_7d,  u.uc_n_cart_updates_7d,  u.uc_n_purchases_7d,
                u.uc_days_since_last_purchase
            FROM '{WORK}/_train_ft.parquet' r
            ASOF LEFT JOIN '{INPUT_RAW}/uc_counters_raw.parquet' u
            ON r.user_id = u.user_id AND r.product_category = u.product_category
            AND r.feature_timestamp > u.ts
        ),
        uc_agg AS (
            SELECT user_id, product_id, request_id,
                MAX(uc_n_views_all) AS uc_n_views_all,
                MAX(uc_n_clicks_all) AS uc_n_clicks_all,
                MAX(uc_n_cart_updates_all) AS uc_n_cart_updates_all,
                MAX(uc_n_purchases_all) AS uc_n_purchases_all,
                MAX(uc_n_views_30d) AS uc_n_views_30d,
                MAX(uc_n_clicks_30d) AS uc_n_clicks_30d,
                MAX(uc_n_cart_updates_30d) AS uc_n_cart_updates_30d,
                MAX(uc_n_purchases_30d) AS uc_n_purchases_30d,
                MAX(uc_n_views_7d) AS uc_n_views_7d,
                MAX(uc_n_clicks_7d) AS uc_n_clicks_7d,
                MAX(uc_n_cart_updates_7d) AS uc_n_cart_updates_7d,
                MAX(uc_n_purchases_7d) AS uc_n_purchases_7d,
                MAX(uc_days_since_last_purchase) AS uc_days_since_last_purchase
            FROM uc_joined
            GROUP BY user_id, product_id, request_id
        )
        SELECT
            b.*,
            uc.uc_n_views_all, uc.uc_n_clicks_all, uc.uc_n_cart_updates_all, uc.uc_n_purchases_all,
            uc.uc_n_views_30d, uc.uc_n_clicks_30d, uc.uc_n_cart_updates_30d, uc.uc_n_purchases_30d,
            uc.uc_n_views_7d,  uc.uc_n_clicks_7d,  uc.uc_n_cart_updates_7d,  uc.uc_n_purchases_7d,
            uc.uc_days_since_last_purchase
        FROM '{WORK}/_train_ui.parquet' b
        LEFT JOIN uc_agg uc
        ON b.user_id = uc.user_id AND b.product_id = uc.product_id AND b.request_id = uc.request_id
    ) TO '{WORK}/counter_features_train.parquet' (FORMAT PARQUET)
""")
print("uc done")

for f in ["_train_ft", "_train_user", "_train_user_item", "_train_ui"]:
    os.remove(WORK / f"{f}.parquet")

print(f"\n=== counter_features_train done ===")
print(pl.read_parquet(WORK / "counter_features_train.parquet").shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

user done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

item done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ui done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

uc done

=== counter_features_train done ===
(1810569, 65)


In [7]:
df_val_ft.write_parquet(WORK / "_val_ft.parquet")

# User counters
duckdb.sql(f"""
    COPY (
        SELECT
            r.user_id, r.product_id, r.request_id, r.split, r.feature_timestamp,
            u.n_views_all, u.n_clicks_all, u.n_cart_updates_all, u.n_purchases_all,
            u.n_views_30d, u.n_clicks_30d, u.n_cart_updates_30d, u.n_purchases_30d,
            u.n_views_7d,  u.n_clicks_7d,  u.n_cart_updates_7d,  u.n_purchases_7d,
            u.n_views_1d,  u.n_clicks_1d,  u.n_cart_updates_1d,  u.n_purchases_1d,
            u.days_since_first_action, u.days_since_last_action
        FROM '{WORK}/_val_ft.parquet' r
        ASOF LEFT JOIN '{INPUT_RAW}/user_counters_raw.parquet' u
        ON r.user_id = u.user_id
        AND r.feature_timestamp > u.ts
    ) TO '{WORK}/_val_user.parquet' (FORMAT PARQUET)
""")
print("user done")

duckdb.sql(f"""
    COPY (
        SELECT
            b.*,
            u.item_n_views_all, u.item_n_clicks_all, u.item_n_cart_updates_all, u.item_n_purchases_all,
            u.item_n_views_30d, u.item_n_clicks_30d, u.item_n_cart_updates_30d, u.item_n_purchases_30d,
            u.item_n_views_7d,  u.item_n_clicks_7d,  u.item_n_cart_updates_7d,  u.item_n_purchases_7d,
            u.item_n_views_1d,  u.item_n_clicks_1d,  u.item_n_cart_updates_1d,  u.item_n_purchases_1d,
            u.item_n_unique_users_all, u.item_n_unique_users_30d, u.item_n_unique_users_7d, u.item_n_unique_users_1d,
            u.item_days_since_first_view, u.item_days_since_last_view
        FROM '{WORK}/_val_user.parquet' b
        ASOF LEFT JOIN '{WORK}/item_counters_raw_v2.parquet' u
        ON b.product_id = u.product_id
        AND b.feature_timestamp > u.ts
    ) TO '{WORK}/_val_user_item.parquet' (FORMAT PARQUET)
""")
print("item done")

duckdb.sql(f"""
    COPY (
        SELECT
            b.*,
            u.ui_n_views_all, u.ui_n_clicks_all, u.ui_n_cart_updates_all, u.ui_n_purchases_all,
            u.ui_days_since_last_action, u.ui_days_since_last_cart_update, u.ui_days_since_last_purchase
        FROM '{WORK}/_val_user_item.parquet' b
        ASOF LEFT JOIN '{INPUT_RAW}/ui_counters_raw.parquet' u
        ON b.user_id = u.user_id AND b.product_id = u.product_id
        AND b.feature_timestamp > u.ts
    ) TO '{WORK}/_val_ui.parquet' (FORMAT PARQUET)
""")
print("ui done")

duckdb.sql(f"""
    COPY (
        WITH uc_joined AS (
            SELECT
                r.user_id, r.product_id, r.request_id,
                u.uc_n_views_all, u.uc_n_clicks_all, u.uc_n_cart_updates_all, u.uc_n_purchases_all,
                u.uc_n_views_30d, u.uc_n_clicks_30d, u.uc_n_cart_updates_30d, u.uc_n_purchases_30d,
                u.uc_n_views_7d,  u.uc_n_clicks_7d,  u.uc_n_cart_updates_7d,  u.uc_n_purchases_7d,
                u.uc_days_since_last_purchase
            FROM '{WORK}/_val_ft.parquet' r
            ASOF LEFT JOIN '{INPUT_RAW}/uc_counters_raw.parquet' u
            ON r.user_id = u.user_id AND r.product_category = u.product_category
            AND r.feature_timestamp > u.ts
        ),
        uc_agg AS (
            SELECT user_id, product_id, request_id,
                MAX(uc_n_views_all) AS uc_n_views_all,
                MAX(uc_n_clicks_all) AS uc_n_clicks_all,
                MAX(uc_n_cart_updates_all) AS uc_n_cart_updates_all,
                MAX(uc_n_purchases_all) AS uc_n_purchases_all,
                MAX(uc_n_views_30d) AS uc_n_views_30d,
                MAX(uc_n_clicks_30d) AS uc_n_clicks_30d,
                MAX(uc_n_cart_updates_30d) AS uc_n_cart_updates_30d,
                MAX(uc_n_purchases_30d) AS uc_n_purchases_30d,
                MAX(uc_n_views_7d) AS uc_n_views_7d,
                MAX(uc_n_clicks_7d) AS uc_n_clicks_7d,
                MAX(uc_n_cart_updates_7d) AS uc_n_cart_updates_7d,
                MAX(uc_n_purchases_7d) AS uc_n_purchases_7d,
                MAX(uc_days_since_last_purchase) AS uc_days_since_last_purchase
            FROM uc_joined
            GROUP BY user_id, product_id, request_id
        )
        SELECT
            b.*,
            uc.uc_n_views_all, uc.uc_n_clicks_all, uc.uc_n_cart_updates_all, uc.uc_n_purchases_all,
            uc.uc_n_views_30d, uc.uc_n_clicks_30d, uc.uc_n_cart_updates_30d, uc.uc_n_purchases_30d,
            uc.uc_n_views_7d,  uc.uc_n_clicks_7d,  uc.uc_n_cart_updates_7d,  uc.uc_n_purchases_7d,
            uc.uc_days_since_last_purchase
        FROM '{WORK}/_val_ui.parquet' b
        LEFT JOIN uc_agg uc
        ON b.user_id = uc.user_id AND b.product_id = uc.product_id AND b.request_id = uc.request_id
    ) TO '{WORK}/counter_features_val.parquet' (FORMAT PARQUET)
""")
print("uc done")

for f in ["_val_ft", "_val_user", "_val_user_item", "_val_ui"]:
    os.remove(WORK / f"{f}.parquet")

print(f"\n=== counter_features_val done ===")
print(pl.read_parquet(WORK / "counter_features_val.parquet").shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

user done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

item done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ui done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

uc done

=== counter_features_val done ===
(323736, 65)


In [8]:
df_test_ft.write_parquet(WORK / "_test_ft.parquet")

# User counters
duckdb.sql(f"""
    COPY (
        SELECT
            r.user_id, r.product_id, r.request_id, r.split, r.feature_timestamp,
            u.n_views_all, u.n_clicks_all, u.n_cart_updates_all, u.n_purchases_all,
            u.n_views_30d, u.n_clicks_30d, u.n_cart_updates_30d, u.n_purchases_30d,
            u.n_views_7d,  u.n_clicks_7d,  u.n_cart_updates_7d,  u.n_purchases_7d,
            u.n_views_1d,  u.n_clicks_1d,  u.n_cart_updates_1d,  u.n_purchases_1d,
            u.days_since_first_action, u.days_since_last_action
        FROM '{WORK}/_test_ft.parquet' r
        ASOF LEFT JOIN '{INPUT_RAW}/user_counters_raw.parquet' u
        ON r.user_id = u.user_id
        AND r.feature_timestamp > u.ts
    ) TO '{WORK}/_test_user.parquet' (FORMAT PARQUET)
""")
print("user done")

duckdb.sql(f"""
    COPY (
        SELECT
            b.*,
            u.item_n_views_all, u.item_n_clicks_all, u.item_n_cart_updates_all, u.item_n_purchases_all,
            u.item_n_views_30d, u.item_n_clicks_30d, u.item_n_cart_updates_30d, u.item_n_purchases_30d,
            u.item_n_views_7d,  u.item_n_clicks_7d,  u.item_n_cart_updates_7d,  u.item_n_purchases_7d,
            u.item_n_views_1d,  u.item_n_clicks_1d,  u.item_n_cart_updates_1d,  u.item_n_purchases_1d,
            u.item_n_unique_users_all, u.item_n_unique_users_30d, u.item_n_unique_users_7d, u.item_n_unique_users_1d,
            u.item_days_since_first_view, u.item_days_since_last_view
        FROM '{WORK}/_test_user.parquet' b
        ASOF LEFT JOIN '{WORK}/item_counters_raw_v2.parquet' u
        ON b.product_id = u.product_id
        AND b.feature_timestamp > u.ts
    ) TO '{WORK}/_test_user_item.parquet' (FORMAT PARQUET)
""")
print("item done")

duckdb.sql(f"""
    COPY (
        SELECT
            b.*,
            u.ui_n_views_all, u.ui_n_clicks_all, u.ui_n_cart_updates_all, u.ui_n_purchases_all,
            u.ui_days_since_last_action, u.ui_days_since_last_cart_update, u.ui_days_since_last_purchase
        FROM '{WORK}/_test_user_item.parquet' b
        ASOF LEFT JOIN '{INPUT_RAW}/ui_counters_raw.parquet' u
        ON b.user_id = u.user_id AND b.product_id = u.product_id
        AND b.feature_timestamp > u.ts
    ) TO '{WORK}/_test_ui.parquet' (FORMAT PARQUET)
""")
print("ui done")

duckdb.sql(f"""
    COPY (
        WITH uc_joined AS (
            SELECT
                r.user_id, r.product_id, r.request_id,
                u.uc_n_views_all, u.uc_n_clicks_all, u.uc_n_cart_updates_all, u.uc_n_purchases_all,
                u.uc_n_views_30d, u.uc_n_clicks_30d, u.uc_n_cart_updates_30d, u.uc_n_purchases_30d,
                u.uc_n_views_7d,  u.uc_n_clicks_7d,  u.uc_n_cart_updates_7d,  u.uc_n_purchases_7d,
                u.uc_days_since_last_purchase
            FROM '{WORK}/_test_ft.parquet' r
            ASOF LEFT JOIN '{INPUT_RAW}/uc_counters_raw.parquet' u
            ON r.user_id = u.user_id AND r.product_category = u.product_category
            AND r.feature_timestamp > u.ts
        ),
        uc_agg AS (
            SELECT user_id, product_id, request_id,
                MAX(uc_n_views_all) AS uc_n_views_all,
                MAX(uc_n_clicks_all) AS uc_n_clicks_all,
                MAX(uc_n_cart_updates_all) AS uc_n_cart_updates_all,
                MAX(uc_n_purchases_all) AS uc_n_purchases_all,
                MAX(uc_n_views_30d) AS uc_n_views_30d,
                MAX(uc_n_clicks_30d) AS uc_n_clicks_30d,
                MAX(uc_n_cart_updates_30d) AS uc_n_cart_updates_30d,
                MAX(uc_n_purchases_30d) AS uc_n_purchases_30d,
                MAX(uc_n_views_7d) AS uc_n_views_7d,
                MAX(uc_n_clicks_7d) AS uc_n_clicks_7d,
                MAX(uc_n_cart_updates_7d) AS uc_n_cart_updates_7d,
                MAX(uc_n_purchases_7d) AS uc_n_purchases_7d,
                MAX(uc_days_since_last_purchase) AS uc_days_since_last_purchase
            FROM uc_joined
            GROUP BY user_id, product_id, request_id
        )
        SELECT
            b.*,
            uc.uc_n_views_all, uc.uc_n_clicks_all, uc.uc_n_cart_updates_all, uc.uc_n_purchases_all,
            uc.uc_n_views_30d, uc.uc_n_clicks_30d, uc.uc_n_cart_updates_30d, uc.uc_n_purchases_30d,
            uc.uc_n_views_7d,  uc.uc_n_clicks_7d,  uc.uc_n_cart_updates_7d,  uc.uc_n_purchases_7d,
            uc.uc_days_since_last_purchase
        FROM '{WORK}/_test_ui.parquet' b
        LEFT JOIN uc_agg uc
        ON b.user_id = uc.user_id AND b.product_id = uc.product_id AND b.request_id = uc.request_id
    ) TO '{WORK}/counter_features_test.parquet' (FORMAT PARQUET)
""")
print("uc done")

for f in ["_test_ft", "_test_user", "_test_user_item", "_test_ui"]:
    os.remove(WORK / f"{f}.parquet")

print(f"\n=== counter_features_test done ===")
print(pl.read_parquet(WORK / "counter_features_test.parquet").shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

user done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

item done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ui done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

uc done

=== counter_features_test done ===
(292826, 65)


In [9]:
!ls -lh /kaggle/working/counter_features_*.parquet

-rw-r--r-- 1 root root  19M Oct 16 09:27 /kaggle/working/counter_features_test.parquet
-rw-r--r-- 1 root root 146M Oct 16 09:26 /kaggle/working/counter_features_train.parquet
-rw-r--r-- 1 root root  20M Oct 16 09:27 /kaggle/working/counter_features_val.parquet


In [10]:
candgen_train = pl.read_parquet(INPUT_INTERACTIONS / "candgen_features_train.parquet")
candgen_val   = pl.read_parquet(INPUT_INTERACTIONS / "candgen_features_val.parquet")
candgen_test  = pl.read_parquet(INPUT_INTERACTIONS / "candgen_features_test.parquet")

print(f"candgen_train: {candgen_train.shape}")
print(f"candgen_val:   {candgen_val.shape}")
print(f"candgen_test:  {candgen_test.shape}")
print(f"\ncandgen_train columns: {candgen_train.columns}")
print(candgen_train.head(3))

# Проверка: сколько уникальных (user_id, product_id) в candgen
print(f"\nunique (user_id, product_id) в candgen_train: {candgen_train.select(['user_id','product_id']).unique().height}")

candgen_train: (15769472, 5)
candgen_val:   (323736, 5)
candgen_test:  (292826, 5)

candgen_train columns: ['user_id', 'product_id', 'als_score', 'v5_score', 'v6_score']
shape: (3, 5)
┌─────────────────────┬──────────────────────┬───────────┬──────────┬───────────┐
│ user_id             ┆ product_id           ┆ als_score ┆ v5_score ┆ v6_score  │
│ ---                 ┆ ---                  ┆ ---       ┆ ---      ┆ ---       │
│ u64                 ┆ u64                  ┆ f32       ┆ f32      ┆ f32       │
╞═════════════════════╪══════════════════════╪═══════════╪══════════╪═══════════╡
│ 7421231689159170748 ┆ 15622873057108008619 ┆ 0.084318  ┆ 1.989193 ┆ 1.568016  │
│ 135364936461809128  ┆ 17225686928570569253 ┆ -0.004961 ┆ 2.108248 ┆ 1.282936  │
│ 7846147652438475366 ┆ 9571392871249924586  ┆ 0.04105   ┆ 1.764622 ┆ -0.513628 │
└─────────────────────┴──────────────────────┴───────────┴──────────┴───────────┘

unique (user_id, product_id) в candgen_train: 3372775


In [11]:
cf_train = pl.read_parquet(WORK / "counter_features_train.parquet")
print(f"cf_train: {cf_train.shape}")

# Убираем feature_timestamp и split — они не нужны как фичи
cf_train = cf_train.drop(["feature_timestamp"])

# Джойним candgen
df_train = cf_train.join(
    candgen_train,
    on=["user_id", "product_id"],
    how="left",
)
print(f"df_train after candgen join: {df_train.shape}")

# Джойним контекст из df_ranking_train_us (source_type, city_name)
context_cols = df_ranking_train_us.select([
    "user_id", "product_id", "request_id",
    "source_type", "city_name", "action_type",
    "position_in_request", "target", "timestamp",
])
df_train = df_train.join(
    context_cols,
    on=["user_id", "product_id", "request_id"],
    how="left",
)
print(f"df_train after context join: {df_train.shape}")

# Добавляем контекстные фичи из timestamp
import datetime
df_train = df_train.with_columns([
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
])

print(f"df_train final: {df_train.shape}")
print(f"columns: {df_train.columns}")
print(f"\nnull_count по ключевым фичам:")
print(df_train.select(["als_score", "v5_score", "v6_score"]).null_count())

cf_train: (1810569, 65)


In [ ]:
### fixing 

In [ ]:
# Проверяем, что файлы на месте
!ls -lh /kaggle/working/*.parquet

In [ ]:
counter_features_train = pl.read_parquet(WORK / "counter_features_train.parquet")
df_ranking_train_us = pl.read_parquet(WORK / "df_ranking_train_us.parquet")
candgen_train = pl.read_parquet(INPUT_INTERACTIONS / "candgen_features_train.parquet")

print("=== counter_features_train ===")
print(f"shape: {counter_features_train.shape}")
print(f"columns: {counter_features_train.columns}")

print("\n=== df_ranking_train_us ===")
print(f"shape: {df_ranking_train_us.shape}")
print(f"columns: {df_ranking_train_us.columns}")

print("\n=== candgen_train ===")
print(f"shape: {candgen_train.shape}")
print(f"columns: {candgen_train.columns}")
print(f"unique (user_id, product_id): {candgen_train.select(['user_id','product_id']).unique().height}")
print(f"total rows: {candgen_train.height}")

In [ ]:
# 1. Дедуплицируем candgen по (user_id, product_id)
candgen_train_dedup = candgen_train.unique(subset=["user_id", "product_id"], keep="first")
print(f"candgen_train_dedup: {candgen_train_dedup.shape}")

# 2. Готовим counter_features (убираем feature_timestamp и split)
cf_train = counter_features_train.drop(["feature_timestamp", "split"])
print(f"cf_train: {cf_train.shape}")

# 3. Джойним candgen
df_train = cf_train.join(
    candgen_train_dedup,
    on=["user_id", "product_id"],
    how="left",
)
print(f"after candgen join: {df_train.shape}")

del cf_train
gc.collect()

# 4. Джойним контекст из df_ranking_train_us
context_cols = df_ranking_train_us.select([
    "user_id", "product_id", "request_id",
    "source_type", "city_name", "action_type",
    "position_in_request", "target", "timestamp",
])
df_train = df_train.join(
    context_cols,
    on=["user_id", "product_id", "request_id"],
    how="left",
)
print(f"after context join: {df_train.shape}")

# 5. Контекстные фичи из timestamp
df_train = df_train.with_columns([
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
])

print(f"\nfinal: {df_train.shape}")
print(f"null_count: {df_train.null_count().sum_horizontal()}")
print(f"\nnulls в candgen scores:")
print(df_train.select(["als_score", "v5_score", "v6_score"]).null_count())

In [ ]:
df_ranking_train_us_dedup = (
    df_ranking_train_us
    .group_by(["user_id", "product_id", "request_id"])
    .agg([
        pl.col("target").max().alias("target"),
        pl.col("position_in_request").min().alias("position_in_request"),
        pl.col("timestamp").max().alias("timestamp"),
        pl.col("source_type").first().alias("source_type"),
        pl.col("city_name").first().alias("city_name"),
        pl.col("action_type").first().alias("action_type"),
    ])
)
print(f"df_ranking_train_us_dedup: {df_ranking_train_us_dedup.shape}")

In [ ]:
# Заново готовим cf_train
cf_train = counter_features_train.drop(["feature_timestamp", "split"])

# Join candgen
df_train = cf_train.join(
    candgen_train_dedup,
    on=["user_id", "product_id"],
    how="left",
)
print(f"after candgen join: {df_train.shape}")

del cf_train
gc.collect()

# Join контекст (уже дедуплицированный)
df_train = df_train.join(
    df_ranking_train_us_dedup,
    on=["user_id", "product_id", "request_id"],
    how="left",
)
print(f"after context join: {df_train.shape}")

# Контекстные фичи из timestamp
df_train = df_train.with_columns([
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
])

print(f"\nfinal df_train: {df_train.shape}")
print(f"target mean: {df_train['target'].mean():.4f}")
print(f"null_count: {df_train.null_count().sum_horizontal()}")

# Сохраняем
df_train.write_parquet(WORK / "train_dataset.parquet")
print("\nSaved train_dataset.parquet")

del df_train
gc.collect()

In [ ]:
# Готовим val
counter_features_val = pl.read_parquet(WORK / "counter_features_val.parquet")
df_ranking_val = pl.read_parquet(INPUT_INTERACTIONS / "df_ranking_val.parquet")
candgen_val = pl.read_parquet(INPUT_INTERACTIONS / "candgen_features_val.parquet")

print(f"counter_features_val: {counter_features_val.shape}")
print(f"df_ranking_val:       {df_ranking_val.shape}")
print(f"candgen_val:          {candgen_val.shape}")

# Дедуплицируем candgen
candgen_val_dedup = candgen_val.unique(subset=["user_id", "product_id"], keep="first")
print(f"candgen_val_dedup: {candgen_val_dedup.shape}")

# Дедуплицируем df_ranking_val
df_ranking_val_dedup = (
    df_ranking_val
    .group_by(["user_id", "product_id", "request_id"])
    .agg([
        pl.col("target").max().alias("target"),
        pl.col("position_in_request").min().alias("position_in_request"),
        pl.col("timestamp").max().alias("timestamp"),
        pl.col("source_type").first().alias("source_type"),
        pl.col("city_name").first().alias("city_name"),
        pl.col("action_type").first().alias("action_type"),
    ])
)
print(f"df_ranking_val_dedup: {df_ranking_val_dedup.shape}")

# Готовим cf_val
cf_val = counter_features_val.drop(["feature_timestamp", "split"])
print(f"cf_val: {cf_val.shape}")

# Join candgen
df_val = cf_val.join(candgen_val_dedup, on=["user_id", "product_id"], how="left")
print(f"after candgen join: {df_val.shape}")

del cf_val
gc.collect()

# Join context
df_val = df_val.join(
    df_ranking_val_dedup,
    on=["user_id", "product_id", "request_id"],
    how="left",
)
print(f"after context join: {df_val.shape}")

# Контекстные фичи из timestamp
df_val = df_val.with_columns([
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
])

print(f"\nfinal df_val: {df_val.shape}")
print(f"target mean: {df_val['target'].mean():.4f}")
print(f"null_count: {df_val.null_count().sum_horizontal()}")

df_val.write_parquet(WORK / "val_dataset.parquet")
print("Saved val_dataset.parquet")

del df_val, df_ranking_val, counter_features_val, candgen_val, df_ranking_val_dedup, candgen_val_dedup
gc.collect()

In [ ]:
df_test = pl.read_parquet(INPUT_INTERACTIONS / "df_test.parquet")
print(f"df_test: {df_test.shape}")
print(f"columns: {df_test.columns}")

In [ ]:
counter_features_test = pl.read_parquet(WORK / "counter_features_test.parquet")
candgen_test = pl.read_parquet(INPUT_INTERACTIONS / "candgen_features_test.parquet")
df_test = pl.read_parquet(INPUT_INTERACTIONS / "df_test.parquet")

print(f"counter_features_test: {counter_features_test.shape}")
print(f"candgen_test:          {candgen_test.shape}")
print(f"df_test:               {df_test.shape}")

# Дедуплицируем candgen
candgen_test_dedup = candgen_test.unique(subset=["user_id", "product_id"], keep="first")
print(f"candgen_test_dedup: {candgen_test_dedup.shape}")

# Дедуплицируем df_test — без position_in_request
df_test_dedup = (
    df_test
    .group_by(["user_id", "product_id", "request_id"])
    .agg([
        pl.col("timestamp").max().alias("timestamp"),
        pl.col("source_type").first().alias("source_type"),
        pl.col("city_name").first().alias("city_name"),
        pl.col("index").first().alias("index"),
    ])
)
print(f"df_test_dedup: {df_test_dedup.shape}")

# cf_test
cf_test = counter_features_test.drop(["feature_timestamp", "split"])
print(f"cf_test: {cf_test.shape}")

# Join candgen
df_test_final = cf_test.join(candgen_test_dedup, on=["user_id", "product_id"], how="left")
print(f"after candgen join: {df_test_final.shape}")

del cf_test
gc.collect()

# Join context
df_test_final = df_test_final.join(
    df_test_dedup,
    on=["user_id", "product_id", "request_id"],
    how="left",
)
print(f"after context join: {df_test_final.shape}")

# Контекстные фичи
df_test_final = df_test_final.with_columns([
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
])

print(f"\nfinal df_test: {df_test_final.shape}")
print(f"null_count: {df_test_final.null_count().sum_horizontal()}")

df_test_final.write_parquet(WORK / "test_dataset.parquet")
print("Saved test_dataset.parquet")

In [ ]:
!ls -lh /kaggle/working/*_dataset.parquet